# Compare to Baselines - Figures

Generates all plots for the baseline comparison experiments across multiple datasets.

## Single Dataset Plot (detailed)

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters to adjust si besoin ---
dataset = "carrot-bowl"             
T = 2000                            
num_runs = 10          
selected_models = ["Sana", "Unidiffuser", "LCM", "Koala", "SDXL-Turbo", "SSD-1B"]

algos = ["Optimal", "Always", "Random", "PAK-UCB",
         "NeMoS 5", "NeMoS 20", "KNN-UCB",
         "LinUCB", "neuronal-s"]

# Configuration générale de la taille des polices
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 10
})

# --- Loading data brutes pickle ---
file_path = f"../experiments/results/compare_to_baselines/data/raw_data_{dataset}_{T}_{num_runs}runs_{len(selected_models)}models.pkl"
with open(file_path, "rb") as f:
    saved = pickle.load(f)

all_OtB = saved["all_o2b"]
all_opr = saved["all_opr"]
budgets_accum = saved["budgets_accum"]

# --- Computing averages pour le plotting ---
avg_OtB = {a: np.mean(np.stack(all_OtB[a]), axis=0) for a in algos}
avg_opr = {a: np.mean(np.stack(all_opr[a]), axis=0) for a in all_opr}
avg_bud = {a: np.mean(np.stack(budgets_accum[a]), axis=0) for a in budgets_accum}

# --- Styles de tracé identiques ---
styles = {
    "Optimal":    {"linestyle": "-.",  "color": "green",     "linewidth": 1.6},
    "Always":     {"linestyle": "--",  "color": "blue",      "linewidth": 1.2},
    "Random":     {"linestyle": "--",  "color": "orange",    "linewidth": 1.2},
    "PAK-UCB":    {"linestyle": "--",  "color": "red",       "linewidth": 1.2},
    "NeMoS 5": {"linestyle": "-",   "color": "darkviolet","linewidth": 2.4},
    "NeMoS 20":{"linestyle": "-",   "color": "indigo",    "linewidth": 2.4},
    "KNN-UCB":    {"linestyle": "--",  "color": "magenta",   "linewidth": 1.2},
    "LinUCB":     {"linestyle": "--",  "color": "gray",      "linewidth": 1.2},
    "neuronal-s": {"linestyle": "--",  "color": "cyan",      "linewidth": 1.2},
}

# --- Creating figures ---
fig, axes = plt.subplots(1, 4, figsize=(26, 6))
window = T // 10
idx = np.linspace(0, T - window, 100, dtype=int)

# (1) Cumulative Regret
ax = axes[0]
for a in algos:
    if a != "Optimal":
        cum_regret = np.cumsum(avg_OtB["Optimal"] - avg_OtB[a])
        ax.plot(np.arange(1, len(cum_regret)+1), cum_regret, label=a, **styles[a])
ax.set_title("Cumulative Regret")
ax.set_xlabel("Iteration")
ax.set_ylabel("Regret")
ax.legend(loc="upper left")
ax.grid(True)

# (2) Sliding-window Avg OPR
ax = axes[1]
for a, v in avg_opr.items():
    if len(v) >= window:
        mov = np.convolve(v, np.ones(window)/window, mode="valid")
        ax.plot(np.arange(window, window+len(mov))[idx], mov[idx], label=a, **styles[a])
ax.set_title(f"{window}-Sliding Avg OPR")
ax.set_xlabel("Iteration")
ax.set_ylabel("Avg OPR")
ax.legend(loc="lower right")
ax.grid(True)

# (3) Budget Consumption
ax = axes[2]
for a, b in avg_bud.items():
    ax.plot(np.arange(1, len(b)+1), b, label=a, **styles[a])
ax.set_title("Budget Consumption")
ax.set_xlabel("Iteration")
ax.set_ylabel("GT Queries")
ax.legend(loc="upper left")
ax.grid(True)

# (4) Sliding-window Avg OtB
ax = axes[3]
for a in algos:
    if len(avg_OtB[a]) >= window:
        mov = np.convolve(avg_OtB[a], np.ones(window)/window, mode="valid")
        ax.plot(np.arange(window, window+len(mov))[idx], mov[idx], label=a, **styles[a])
ax.set_title(f"{window}-Sliding Avg OtB")
ax.set_xlabel("Iteration")
ax.set_ylabel("Avg OtB")
ax.legend(loc="lower right")
ax.grid(True)

plt.tight_layout()
os.makedirs(f"plots/compare_to_baselines", exist_ok=True)
plt.savefig(f"plots/compare_to_baselines/{dataset}_{T}_{num_runs}runs_{len(selected_models)}models.pdf", dpi=600)
plt.show()

## Combined OtB (Multiple Datasets)

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# --- Specific parameters à chaque dataset ---
dataset_params = {
    "ms-coco":      {"T": 5000, "num_runs": 5},
    "flickr":       {"T": 3000, "num_runs": 10},
    "flowers":      {"T": 1000, "num_runs": 20},
    "carrot-bowl":  {"T": 2000, "num_runs": 10}
}
# --- Mapping noms affichés ---
pretty_names = {
    "ms-coco": "MS-COCO",
    "flickr": "Flickr",
    "flowers": "Flowers",
    "carrot-bowl": "Carrot-bowl"
}

selected_models = ["Sana", "Unidiffuser", "LCM", "Koala", "SDXL-Turbo", "SSD-1B"]

algos = ["Optimal", "Always", "Random", "PAK-UCB",
         "NeMoS 5", "NeMoS 20", "KNN-UCB",
         "LinUCB", "neuronal-s"]


# --- Styles globaux (grandes polices) ---
plt.rcParams.update({
    'font.size': 26,
    'axes.titlesize': 26,
    'axes.labelsize': 24,
    'xtick.labelsize': 26,
    'ytick.labelsize': 24,
    'legend.fontsize': 26
})

styles = {
    "Optimal":    {"linestyle": "-.",  "color": "green",     "linewidth": 2.0},
    "Always":     {"linestyle": "--",  "color": "blue",      "linewidth": 1.8},
    "Random":     {"linestyle": "--",  "color": "orange",    "linewidth": 1.8},
    "PAK-UCB":    {"linestyle": "--",  "color": "red",       "linewidth": 1.8},
    "NeMoS 5":    {"linestyle": "-",   "color": "darkviolet","linewidth": 2.6},
    "NeMoS 20":   {"linestyle": "-",   "color": "indigo",    "linewidth": 2.6},
    "KNN-UCB":    {"linestyle": "--",  "color": "magenta",   "linewidth": 1.8},
    "LinUCB":     {"linestyle": "--",  "color": "gray",      "linewidth": 1.8},
    "neuronal-s": {"linestyle": "--",  "color": "cyan",      "linewidth": 1.8},
}

# --- Figure multi-datasets avec Y partagé ---
fig, axes = plt.subplots(1, 4, figsize=(30, 7), sharey=True)

legend_handles, legend_labels = [], []

for i, (dataset, params) in enumerate(dataset_params.items()):
    T = params["T"]
    num_runs = params["num_runs"]
    window = T // 10
    idx = np.linspace(0, T - window, 100, dtype=int)

    file_path = f"../experiments/results/compare_to_baselines/data/raw_data_{dataset}_{T}_{num_runs}runs_{len(selected_models)}models.pkl"
    with open(file_path, "rb") as f:
        saved = pickle.load(f)

    all_OtB = saved["all_o2b"]

    # Moyenne sur runs
    alias = {
        "NeMoS 5": "NeMoS 5",
        "NeMoS 20": "NeMoS 20",
    }

    avg_OtB = {}
    for a in all_OtB:
        if len(all_OtB[a]) == 0:
            continue
        name = alias.get(a, a)
        avg_OtB[name] = np.mean(np.stack(all_OtB[a]), axis=0)

    # Plot du sliding avg OtB
    ax = axes[i]
    for a in algos:
        v = avg_OtB.get(a, None)
        if v is not None and len(v) >= window:
            mov = np.convolve(v, np.ones(window)/window, mode="valid")
            ln, = ax.plot(np.arange(window, window+len(mov))[idx], mov[idx],
                          label=a, **styles.get(a, {}))
            if i == 0:  # collecte une seule fois
                legend_handles.append(ln)
                legend_labels.append(a)

    # Titre en haut : taille de fenêtre
    ax.set_title(f"{window}-Sliding Avg OtB")

    # Titres bas (datasets) avec noms "propres"
    ax.annotate(pretty_names.get(dataset, dataset),
                xy=(0.5, -0.30), xycoords='axes fraction',
                ha='center', va='top', fontsize=34)

    ax.grid(True)

# --- Étiquette Y commune ---
fig.supylabel("OtB", x=0.02, y=0.64)

# --- Marges + légende commune centrée en bas ---
fig.subplots_adjust(bottom=0.32, top=0.90, left=0.06, right=0.98, wspace=0.08)

fig.legend(legend_handles, legend_labels,
           loc='lower center', bbox_to_anchor=(0.50, 0.13),
           ncol=len(legend_labels), frameon=False)

os.makedirs(f"plots/compare_to_baselines", exist_ok=True)
plt.savefig(f"plots/compare_to_baselines/combined_OtB.pdf",
            dpi=600, bbox_inches="tight")
plt.show()

## Combined OtB with Query Optimal

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# --- Specific parameters à chaque dataset ---
dataset_params = {
    "ms-coco":      {"T": 5000, "num_runs": 5},
    "flickr":       {"T": 3000, "num_runs": 10},
    "flowers":      {"T": 1000, "num_runs": 20},
    "carrot-bowl":  {"T": 2000, "num_runs": 10}
}
# --- Mapping noms affichés ---
pretty_names = {
    "ms-coco": "MS-COCO",
    "flickr": "Flickr",
    "flowers": "Flowers",
    "carrot-bowl": "Carrot-bowl"
}

selected_models = ["Sana", "Unidiffuser", "LCM", "Koala", "SDXL-Turbo", "SSD-1B"]

algos = ["Optimal", "Always", "Random", "PAK-UCB",
         "NeMoS 5", "NeMoS 20", "KNN-UCB",
         "LinUCB", "neuronal-s"]

# --- Styles globaux : + grand partout ---
plt.rcParams.update({
    'font.size': 22,
    'axes.titlesize': 24,
    'axes.labelsize': 20,
    'xtick.labelsize': 22,
    'ytick.labelsize': 22,
    'legend.fontsize': 20
})

styles = {
    "Optimal":    {"linestyle": "-.",  "color": "green",     "linewidth": 2.0},
    "Always":     {"linestyle": "--",  "color": "blue",      "linewidth": 1.8},
    "Random":     {"linestyle": "--",  "color": "orange",    "linewidth": 1.8},
    "PAK-UCB":    {"linestyle": "--",  "color": "red",       "linewidth": 1.8},
    "NeMoS 5": {"linestyle": "-",   "color": "darkviolet","linewidth": 2.6},
    "NeMoS 20":{"linestyle": "-",   "color": "indigo",    "linewidth": 2.6},
    "KNN-UCB":    {"linestyle": "--",  "color": "magenta",   "linewidth": 1.8},
    "LinUCB":     {"linestyle": "--",  "color": "gray",      "linewidth": 1.8},
    "neuronal-s": {"linestyle": "--",  "color": "cyan",      "linewidth": 1.8},
}

def query_mask_from_budget(b):
    b = np.asarray(b, dtype=float)
    inc = np.diff(b, prepend=b[0])
    return inc > 1e-12

# --- Figure multi-datasets avec Y partagé ---
fig, axes = plt.subplots(1, 4, figsize=(30, 7), sharey=True)

legend_handles, legend_labels = [], []

for i, (dataset, params) in enumerate(dataset_params.items()):
    T = params["T"]
    num_runs = params["num_runs"]
    window = T // 10
    idx = np.linspace(0, T - window, 100, dtype=int)

    file_path = f"../experiments/results/compare_to_baselines/data/raw_data_{dataset}_{T}_{num_runs}runs_{len(selected_models)}models.pkl"
    with open(file_path, "rb") as f:
        saved = pickle.load(f)

    all_OtB = saved["all_o2b"]
    budgets_accum = saved["budgets_accum"]

    if "Optimal" not in all_OtB:
        raise KeyError("Série 'Optimal' absente dans all_o2b — nécessaire pour le remplacement OtB.")

    # Copie ajustable
    all_OtB_adj = {a: [np.array(run, copy=True) for run in runs] for a, runs in all_OtB.items()}

    # Remplacement OtB -> Optimal lors des queries
    for a in algos:
        if a == "Optimal" or a not in all_OtB_adj or a not in budgets_accum:
            continue
        runs_a = all_OtB_adj[a]
        runs_opt = all_OtB_adj["Optimal"]
        buds_a = budgets_accum[a]

        for r in range(min(len(runs_a), len(runs_opt), len(buds_a))):
            series = runs_a[r]
            opt_series = np.asarray(runs_opt[r])
            bud = np.asarray(buds_a[r])
            L = min(len(series), len(opt_series), len(bud))
            if L == 0:
                continue
            series = series[:L]
            opt_series = opt_series[:L]
            bud = bud[:L]

            mask = query_mask_from_budget(bud)
            if mask.any():
                series[mask] = opt_series[mask]
                runs_a[r] = series

    # Moyenne sur runs après ajustement
    avg_OtB = {a: np.mean(np.stack(all_OtB_adj[a]), axis=0) for a in all_OtB_adj if len(all_OtB_adj[a]) > 0}

    # Plot du sliding avg OtB
    ax = axes[i]
    for a in algos:
        v = avg_OtB.get(a, None)
        if v is not None and len(v) >= window:
            mov = np.convolve(v, np.ones(window)/window, mode="valid")
            ln, = ax.plot(np.arange(window, window+len(mov))[idx], mov[idx],
                          label=a, **styles.get(a, {}))
            if i == 0:  # collecte une seule fois
                legend_handles.append(ln)
                legend_labels.append(a)

    # Titre en haut : taille de fenêtre
    ax.set_title(f"{window}-Sliding Avg OtB")

    # Titres bas (datasets) avec noms "propres"
    ax.annotate(pretty_names.get(dataset, dataset),
                xy=(0.5, -0.30), xycoords='axes fraction',
                ha='center', va='top', fontsize=24)

    ax.grid(True)

# --- Étiquettes communes d'axes ---
fig.supylabel("OtB", x=0.02, y=0.64)

# --- Ajustement des marges et de la légende ---
fig.subplots_adjust(bottom=0.30, top=0.90, left=0.06, right=0.98, wspace=0.08)

fig.legend(legend_handles, legend_labels,
           loc='lower center', bbox_to_anchor=(0.50, 0.13),
           ncol=len(legend_labels), frameon=False)

os.makedirs(f"plots/compare_to_baselines", exist_ok=True)
plt.savefig(f"plots/compare_to_baselines/combined_OtB_query_optimal.pdf",
            dpi=600, bbox_inches="tight")
plt.show()

## Combined OPR (Multiple Datasets)

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# --- Specific parameters à chaque dataset ---
dataset_params = {
    "ms-coco":      {"T": 5000, "num_runs": 5},
    "flickr":       {"T": 3000, "num_runs": 10},
    "flowers":      {"T": 1000, "num_runs": 20},
    "carrot-bowl":  {"T": 2000, "num_runs": 10}
}
# --- Mapping noms affichés ---
pretty_names = {
    "ms-coco": "MS-COCO",
    "flickr": "Flickr",
    "flowers": "Flowers",
    "carrot-bowl": "Carrot-bowl"
}

selected_models = ["Sana", "Unidiffuser", "LCM", "Koala", "SDXL-Turbo", "SSD-1B"]

algos = ["Optimal", "Always", "Random", "PAK-UCB",
         "NeMoS 5", "NeMoS 20", "KNN-UCB",
         "LinUCB", "neuronal-s"]

# --- Styles globaux (grandes polices) ---
plt.rcParams.update({
    'font.size': 26,
    'axes.titlesize': 26,
    'axes.labelsize': 24,
    'xtick.labelsize': 26,
    'ytick.labelsize': 24,
    'legend.fontsize': 26
})

styles = {
    "Optimal":    {"linestyle": "-.",  "color": "green",     "linewidth": 2.0},
    "Always":     {"linestyle": "--",  "color": "blue",      "linewidth": 1.8},
    "Random":     {"linestyle": "--",  "color": "orange",    "linewidth": 1.8},
    "PAK-UCB":    {"linestyle": "--",  "color": "red",       "linewidth": 1.8},
    "NeMoS 5": {"linestyle": "-",   "color": "darkviolet","linewidth": 2.6},
    "NeMoS 20":{"linestyle": "-",   "color": "indigo",    "linewidth": 2.6},
    "KNN-UCB":    {"linestyle": "--",  "color": "magenta",   "linewidth": 1.8},
    "LinUCB":     {"linestyle": "--",  "color": "gray",      "linewidth": 1.8},
    "neuronal-s": {"linestyle": "--",  "color": "cyan",      "linewidth": 1.8},
}

# --- Figure multi-datasets avec Y partagé ---
fig, axes = plt.subplots(1, 4, figsize=(30, 7), sharey=True)

legend_handles, legend_labels = [], []

for i, (dataset, params) in enumerate(dataset_params.items()):
    T = params["T"]
    num_runs = params["num_runs"]
    window = T // 10
    idx = np.linspace(0, T - window, 100, dtype=int)

    file_path = f"../experiments/results/compare_to_baselines/data/raw_data_{dataset}_{T}_{num_runs}runs_{len(selected_models)}models.pkl"
    with open(file_path, "rb") as f:
        saved = pickle.load(f)

    all_opr = saved["all_opr"]

    # Moyenne sur runs
    avg_OPR = {a: np.mean(np.stack(all_opr[a]), axis=0)
               for a in all_opr if len(all_opr[a]) > 0}

    # Plot sliding avg OPR
    ax = axes[i]
    for a in algos:
        v = avg_OPR.get(a, None)
        if v is not None and len(v) >= window:
            mov = np.convolve(v, np.ones(window)/window, mode="valid")
            ln, = ax.plot(np.arange(window, window+len(mov))[idx], mov[idx],
                          label=a, **styles.get(a, {}))
            if i == 0:
                legend_handles.append(ln)
                legend_labels.append(a)

    ax.set_title(f"{window}-Sliding Avg OPR")
    ax.annotate(pretty_names.get(dataset, dataset),
                xy=(0.5, -0.30), xycoords='axes fraction',
                ha='center', va='top', fontsize=34)
    ax.grid(True)

# --- Étiquette Y commune ---
fig.supylabel("OPR", x=0.02, y=0.64)

# --- Marges + légende commune centrée en bas ---
fig.subplots_adjust(bottom=0.32, top=0.90, left=0.06, right=0.98, wspace=0.08)

fig.legend(legend_handles, legend_labels,
           loc='lower center', bbox_to_anchor=(0.50, 0.13),
           ncol=len(legend_labels), frameon=False)

os.makedirs(f"plots/compare_to_baselines", exist_ok=True)
plt.savefig(f"plots/compare_to_baselines/combined_OPR.pdf",
            dpi=600, bbox_inches="tight")
plt.show()

## Combined Budget (Multiple Datasets)

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# --- Specific parameters à chaque dataset ---
dataset_params = {
    "ms-coco":      {"T": 5000, "num_runs": 5},
    "flickr":       {"T": 3000, "num_runs": 10},
    "flowers":      {"T": 1000, "num_runs": 20},
    "carrot-bowl":  {"T": 2000, "num_runs": 10}
}
# --- Mapping noms affichés ---
pretty_names = {
    "ms-coco": "MS-COCO",
    "flickr": "Flickr",
    "flowers": "Flowers",
    "carrot-bowl": "Carrot-bowl"
}

selected_models = ["Sana", "Unidiffuser", "LCM", "Koala", "SDXL-Turbo", "SSD-1B"]

algos = ["Optimal", "Always", "Random", "PAK-UCB",
         "NeMoS 5", "NeMoS 20", "KNN-UCB",
         "LinUCB", "neuronal-s"]

# --- Styles globaux ---
plt.rcParams.update({
    'font.size': 26,
    'axes.titlesize': 26,
    'axes.labelsize': 24,
    'xtick.labelsize': 24,
    'ytick.labelsize': 18,   # police réduite
    'legend.fontsize': 26
})

styles = {
    "Optimal":    {"linestyle": "-.",  "color": "green",     "linewidth": 2.0},
    "Always":     {"linestyle": "--",  "color": "blue",      "linewidth": 1.8},
    "Random":     {"linestyle": "--",  "color": "orange",    "linewidth": 1.8},
    "PAK-UCB":    {"linestyle": "--",  "color": "red",       "linewidth": 1.8},
    "NeMoS 5": {"linestyle": "-",   "color": "darkviolet","linewidth": 2.6},
    "NeMoS 20":{"linestyle": "-",   "color": "indigo",    "linewidth": 2.6},
    "KNN-UCB":    {"linestyle": "--",  "color": "magenta",   "linewidth": 1.8},
    "LinUCB":     {"linestyle": "--",  "color": "gray",      "linewidth": 1.8},
    "neuronal-s": {"linestyle": "--",  "color": "cyan",      "linewidth": 1.8},
}

# --- Figure multi-datasets ---
fig, axes = plt.subplots(1, 4, figsize=(32, 7), sharey=False)

legend_handles, legend_labels = [], []

for i, (dataset, params) in enumerate(dataset_params.items()):
    T = params["T"]
    num_runs = params["num_runs"]

    file_path = f"../experiments/results/compare_to_baselines/data/raw_data_{dataset}_{T}_{num_runs}runs_{len(selected_models)}models.pkl"
    with open(file_path, "rb") as f:
        saved = pickle.load(f)

    budgets_accum = saved.get("budgets_accum", {})

    # Moyenne sur runs
    avg_bud = {a: np.mean(np.stack(budgets_accum[a]), axis=0)
               for a in budgets_accum if len(budgets_accum[a]) > 0}

    # Plot Budget
    ax = axes[i]
    for a in algos:
        series = avg_bud.get(a, None)
        if series is not None and len(series) > 0:
            ln, = ax.plot(np.arange(1, len(series)+1), series,
                          label=a, **styles.get(a, {}))
            if i == 0:
                legend_handles.append(ln)
                legend_labels.append(a)

    # Nom du dataset en bas
    ax.annotate(pretty_names.get(dataset, dataset),
                xy=(0.5, -0.30), xycoords='axes fraction',
                ha='center', va='top', fontsize=34)

    # Label Y uniquement sur le premier subplot
    if i == 0:
        ax.set_ylabel("GT Queries")

    ax.grid(True)

# --- Marges + légende commune centrée en bas ---
fig.subplots_adjust(bottom=0.32, top=0.92, left=0.06, right=0.98, wspace=0.18)

fig.legend(legend_handles, legend_labels,
           loc='lower center', bbox_to_anchor=(0.50, 0.13),
           ncol=len(legend_labels), frameon=False)

os.makedirs(f"plots/compare_to_baselines", exist_ok=True)
plt.savefig(f"plots/compare_to_baselines/combined_Budget.pdf",
            dpi=600, bbox_inches="tight")
plt.show()